# Aircraft Data Simulator

This notebook generates synthetic aircraft telemetry for a predictive maintenance demo. It models 10 airlines, diverse aircraft fleets, realistic flight phase behavior, sensor-level telemetry, and injected maintenance anomalies.

The notebook also creates Unity Catalog tables for historical telemetry, flights metadata, and aircraft registry data, plus a lightweight notebook-hosted API that can be used to emulate Zerobus ingestion and a live streaming simulator for append-only telemetry generation.


In [0]:
import json
import math
import os
import random
import socket
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, date, datetime, timedelta
from itertools import cycle
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)


def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)

DEFAULT_CATALOG = "ayman_el_ghazali"
try:
    dbutils.widgets.text("catalog_name", DEFAULT_CATALOG)
    widget_catalog = dbutils.widgets.get("catalog_name").strip()
except Exception:
    widget_catalog = ""

catalog_name = widget_catalog or spark.sql("SELECT current_catalog()").first()[0]
schema_name = "aircraft_maintenance"

NUM_AIRLINES = 10
FLIGHTS_HISTORICAL = 1000
RECORDS_PER_FLIGHT = 1000
ANOMALY_RATE = 0.10
RANDOM_SEED = 42

current_catalog = spark.sql("SELECT current_catalog()").first()[0]
if not catalog_name:
    catalog_name = current_catalog

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog_name}`")
except Exception as create_catalog_error:
    print(f"Falling back to current catalog '{current_catalog}' because '{catalog_name}' could not be created or used: {create_catalog_error}")
    catalog_name = current_catalog

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

TABLE_NAMESPACE = f"`{catalog_name}`.`{schema_name}`"
TABLE_NAMESPACE_FQN = f"{catalog_name}.{schema_name}"
RAW_TABLE = f"{TABLE_NAMESPACE}.`raw_flight_telemetry`"
RAW_TABLE_FQN = f"{TABLE_NAMESPACE_FQN}.raw_flight_telemetry"
FLIGHTS_METADATA_TABLE = f"{TABLE_NAMESPACE}.`flights_metadata`"
FLIGHTS_METADATA_TABLE_FQN = f"{TABLE_NAMESPACE_FQN}.flights_metadata"
AIRCRAFT_REGISTRY_TABLE = f"{TABLE_NAMESPACE}.`aircraft_registry`"
AIRCRAFT_REGISTRY_TABLE_FQN = f"{TABLE_NAMESPACE_FQN}.aircraft_registry"
API_STAGING_TABLE = f"{TABLE_NAMESPACE}.`staging_flight_telemetry_api`"
API_STAGING_TABLE_FQN = f"{TABLE_NAMESPACE_FQN}.staging_flight_telemetry_api"

SIMULATOR_CONFIG = {
    "catalog_name": catalog_name,
    "schema_name": schema_name,
    "num_airlines": NUM_AIRLINES,
    "historical_flights": FLIGHTS_HISTORICAL,
    "records_per_flight": RECORDS_PER_FLIGHT,
    "anomaly_rate": ANOMALY_RATE,
    "random_seed": RANDOM_SEED,
}

config_pdf = pd.DataFrame([SIMULATOR_CONFIG])
display(config_pdf)
print(f"Target schema ready: {TABLE_NAMESPACE_FQN}")


{"ts": "2026-08-31 14:22:55.568", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"PERMISSION_DENIED: User does not have CREATE CATALOG on Metastore 'metastore_aws_us_west_2'.\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {created_time:\"2026-08-31T14:22:55.56755443+00:00\", grpc_status:13, grpc_message:\"PERMISSION_DENIED: User does not have CREATE CATALOG on Metastore \\'metastore_aws_us_west_2\\'.\"}\"\n>", "stacktrace": [{"class": null, "method": "_execute_and_fetch_as_iterator", "file": "/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py", "line": "2046"}, {"class": null, "method": "__next__", "file": "<frozen _collections_abc>", "line": "356"}, {"class": null, "method": "send", "file": "/databricks/python/lib/python3.12/

Falling back to current catalog 'genie_zeroops_mfg_catalog' because 'ayman_el_ghazali' could not be created or used: (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_DENIED: User does not have CREATE CATALOG on Metastore 'metastore_aws_us_west_2'.

JVM stacktrace:
com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException
	at com.databricks.sql.managedcatalog.client.ErrorDetailsHandlerImpl.wrapServiceException(ErrorDetailsHandler.scala:119)
	at com.databricks.sql.managedcatalog.client.ErrorDetailsHandlerImpl.wrapServiceException$(ErrorDetailsHandler.scala:88)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.wrapServiceException(ManagedCatalogClientImpl.scala:44)
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.recordAndWrapExceptionBase(ManagedCatalogClientImpl.scala:8507)
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.recordAndWrapException(ManagedCatalogClientImpl.scala:8457)
	at com.databricks

catalog_name,schema_name,num_airlines,historical_flights,records_per_flight,anomaly_rate,random_seed
genie_zeroops_mfg_catalog,aircraft_maintenance,10,1000,1000,0.1,42


Target schema ready: genie_zeroops_mfg_catalog.aircraft_maintenance


In [0]:
AIRLINES = [
    "SkyWest Global",
    "Atlas Pacific",
    "Northern Star Airways",
    "Meridian Airlines",
    "Falcon Jet Airways",
    "TransOcean Air",
    "Summit Aviation",
    "Horizon Express",
    "Continental Wings",
    "Aurora Airlines",
]

AIRCRAFT_TYPES = [
    "B737-800",
    "A320neo",
    "B777-300ER",
    "A350-900",
    "B787-9",
    "A321neo",
    "B767-300",
    "A330-900",
    "E190-E2",
    "CRJ-900",
]

AIRPORTS = {
    "JFK": {"city": "New York", "lat": 40.6413, "lon": -73.7781},
    "LAX": {"city": "Los Angeles", "lat": 33.9416, "lon": -118.4085},
    "SFO": {"city": "San Francisco", "lat": 37.6213, "lon": -122.3790},
    "SEA": {"city": "Seattle", "lat": 47.4502, "lon": -122.3088},
    "ORD": {"city": "Chicago", "lat": 41.9742, "lon": -87.9073},
    "ATL": {"city": "Atlanta", "lat": 33.6407, "lon": -84.4277},
    "DFW": {"city": "Dallas", "lat": 32.8998, "lon": -97.0403},
    "MIA": {"city": "Miami", "lat": 25.7959, "lon": -80.2870},
    "DEN": {"city": "Denver", "lat": 39.8561, "lon": -104.6737},
    "BOS": {"city": "Boston", "lat": 42.3656, "lon": -71.0096},
    "LHR": {"city": "London", "lat": 51.4700, "lon": -0.4543},
    "CDG": {"city": "Paris", "lat": 49.0097, "lon": 2.5479},
    "FRA": {"city": "Frankfurt", "lat": 50.0379, "lon": 8.5622},
    "AMS": {"city": "Amsterdam", "lat": 52.3105, "lon": 4.7683},
    "MAD": {"city": "Madrid", "lat": 40.4983, "lon": -3.5676},
    "DXB": {"city": "Dubai", "lat": 25.2532, "lon": 55.3657},
    "SIN": {"city": "Singapore", "lat": 1.3644, "lon": 103.9915},
    "HND": {"city": "Tokyo", "lat": 35.5494, "lon": 139.7798},
    "NRT": {"city": "Tokyo Narita", "lat": 35.7720, "lon": 140.3929},
    "SYD": {"city": "Sydney", "lat": -33.9399, "lon": 151.1753},
    "YYZ": {"city": "Toronto", "lat": 43.6777, "lon": -79.6248},
    "GRU": {"city": "Sao Paulo", "lat": -23.4356, "lon": -46.4731},
}

ROUTE_CATALOG = [
    {"origin": "JFK", "destination": "LAX"},
    {"origin": "LAX", "destination": "SEA"},
    {"origin": "ORD", "destination": "DFW"},
    {"origin": "ATL", "destination": "MIA"},
    {"origin": "SFO", "destination": "DEN"},
    {"origin": "BOS", "destination": "ATL"},
    {"origin": "LHR", "destination": "NRT"},
    {"origin": "CDG", "destination": "DXB"},
    {"origin": "FRA", "destination": "SIN"},
    {"origin": "AMS", "destination": "MAD"},
    {"origin": "SYD", "destination": "SIN"},
    {"origin": "YYZ", "destination": "JFK"},
    {"origin": "GRU", "destination": "MIA"},
    {"origin": "HND", "destination": "SFO"},
    {"origin": "DXB", "destination": "LHR"},
    {"origin": "SEA", "destination": "ORD"},
    {"origin": "MIA", "destination": "BOS"},
    {"origin": "DEN", "destination": "LAX"},
    {"origin": "MAD", "destination": "CDG"},
    {"origin": "SIN", "destination": "SYD"},
]


def great_circle_nm(origin_code: str, destination_code: str) -> float:
    origin = AIRPORTS[origin_code]
    destination = AIRPORTS[destination_code]
    lat1 = math.radians(origin["lat"])
    lon1 = math.radians(origin["lon"])
    lat2 = math.radians(destination["lat"])
    lon2 = math.radians(destination["lon"])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return 3440.065 * c


def generate_tail_numbers(total_needed: int, rng: np.random.Generator) -> List[str]:
    tail_numbers: List[str] = []
    seen = set()
    while len(tail_numbers) < total_needed:
        value = f"N{int(rng.integers(10000, 99999))}"
        if value not in seen:
            seen.add(value)
            tail_numbers.append(value)
    return tail_numbers


rng_fleet = np.random.default_rng(RANDOM_SEED)
AIRLINE_ROUTE_MAP: Dict[str, List[Dict[str, str]]] = {}
for airline_index, airline in enumerate(AIRLINES):
    route_indices = [(airline_index * 2 + offset) % len(ROUTE_CATALOG) for offset in range(8)]
    AIRLINE_ROUTE_MAP[airline] = [ROUTE_CATALOG[idx] for idx in route_indices]

fleet_sizes = {airline: int(rng_fleet.integers(20, 51)) for airline in AIRLINES}
tail_number_pool = generate_tail_numbers(sum(fleet_sizes.values()), rng_fleet)
tail_number_iter = iter(tail_number_pool)

fleet_records: List[Dict[str, Any]] = []
AIRLINE_FLEET_MAP: Dict[str, List[Dict[str, Any]]] = {}

for airline_index, airline in enumerate(AIRLINES):
    airline_fleet: List[Dict[str, Any]] = []
    for aircraft_idx in range(fleet_sizes[airline]):
        aircraft_age_years = int(rng_fleet.integers(1, 26))
        manufacture_date = date.today() - timedelta(days=int(365.25 * aircraft_age_years + rng_fleet.integers(0, 365)))
        last_maintenance_date = date.today() - timedelta(days=int(rng_fleet.integers(15, 180)))
        next_scheduled_maintenance = last_maintenance_date + timedelta(days=180)
        total_flight_hours = round(float(aircraft_age_years * rng_fleet.uniform(1800, 3600)), 1)
        record = {
            "airline": airline,
            "tail_number": next(tail_number_iter),
            "aircraft_type": AIRCRAFT_TYPES[(airline_index + aircraft_idx) % len(AIRCRAFT_TYPES)],
            "aircraft_age_years": aircraft_age_years,
            "home_hub": AIRLINE_ROUTE_MAP[airline][0]["origin"],
            "manufacture_date": manufacture_date,
            "total_flight_hours": total_flight_hours,
            "last_maintenance_date": last_maintenance_date,
            "next_scheduled_maintenance": next_scheduled_maintenance,
        }
        fleet_records.append(record)
        airline_fleet.append(record)
    AIRLINE_FLEET_MAP[airline] = airline_fleet

AIRCRAFT_REGISTRY_PDF = pd.DataFrame(fleet_records)
AIRCRAFT_LOOKUP = {row["tail_number"]: row for row in fleet_records}

fleet_summary_pdf = (
    AIRCRAFT_REGISTRY_PDF.groupby(["airline", "aircraft_type"], as_index=False)
    .size()
    .rename(columns={"size": "aircraft_count"})
)

display(fleet_summary_pdf)
print(f"Generated fleet registry for {len(AIRCRAFT_REGISTRY_PDF):,} aircraft across {len(AIRLINES)} airlines.")


airline,aircraft_type,aircraft_count
Atlas Pacific,A320neo,5
Atlas Pacific,A321neo,4
Atlas Pacific,A330-900,4
Atlas Pacific,A350-900,5
Atlas Pacific,B737-800,4
Atlas Pacific,B767-300,4
Atlas Pacific,B777-300ER,5
Atlas Pacific,B787-9,4
Atlas Pacific,CRJ-900,4
Atlas Pacific,E190-E2,4


Generated fleet registry for 328 aircraft across 10 airlines.


In [0]:
def make_param(
    param_name: str,
    unit: str,
    system_group: str,
    normal_min: float,
    normal_max: float,
    anomaly_min: float,
    anomaly_max: float,
    description: str,
) -> Dict[str, Any]:
    return {
        "param_name": param_name,
        "unit": unit,
        "system_group": system_group,
        "normal_min": normal_min,
        "normal_max": normal_max,
        "anomaly_min": anomaly_min,
        "anomaly_max": anomaly_max,
        "description": description,
    }


PARAMETER_REGISTRY: List[Dict[str, Any]] = []

engine_templates = {
    "egt": ("degC", 380, 930, 320, 1050, "Exhaust gas temperature"),
    "n1_speed": ("pct", 18, 102, 10, 108, "Low-pressure spool speed"),
    "n2_speed": ("pct", 52, 106, 40, 112, "High-pressure spool speed"),
    "oil_pressure": ("psi", 28, 78, 5, 95, "Engine oil pressure"),
    "oil_temp": ("degC", 65, 145, 20, 190, "Engine oil temperature"),
    "fuel_flow": ("kg_hr", 250, 6200, 100, 7500, "Per-engine fuel flow"),
    "vibration_n1": ("ips", 0.02, 1.4, 0.0, 4.0, "N1 shaft vibration amplitude"),
    "vibration_n2": ("ips", 0.02, 1.2, 0.0, 3.5, "N2 shaft vibration amplitude"),
    "epr": ("ratio", 1.0, 2.2, 0.8, 2.5, "Engine pressure ratio"),
    "ff_ratio": ("ratio", 0.75, 1.25, 0.4, 1.8, "Fuel-flow efficiency ratio"),
    "bleed_pressure": ("psi", 18, 52, 5, 70, "Engine bleed air pressure"),
    "bleed_temp": ("degC", 120, 260, 60, 340, "Engine bleed air temperature"),
    "thrust_output": ("pct", 8, 100, 0, 110, "Commanded thrust output"),
    "starter_valve_pos": ("pct", 0, 100, 0, 100, "Starter valve position"),
    "reverser_pos": ("pct", 0, 100, 0, 100, "Thrust reverser position"),
}

for engine_id in (1, 2):
    for name, spec in engine_templates.items():
        PARAMETER_REGISTRY.append(
            make_param(
                param_name=f"{name}_eng{engine_id}",
                unit=spec[0],
                system_group="ENGINE",
                normal_min=spec[1],
                normal_max=spec[2],
                anomaly_min=spec[3],
                anomaly_max=spec[4],
                description=f"Engine {engine_id} {spec[5].lower()}",
            )
        )

flight_params = [
    ("altitude", "ft", "FLIGHT", 0, 41000, 0, 45000, "Pressure altitude"),
    ("indicated_airspeed", "kts", "FLIGHT", 0, 360, 0, 420, "Indicated airspeed"),
    ("ground_speed", "kts", "FLIGHT", 0, 550, 0, 620, "Ground speed"),
    ("mach_number", "mach", "FLIGHT", 0, 0.88, 0, 0.95, "Mach number"),
    ("vertical_speed", "fpm", "FLIGHT", -4500, 4500, -7000, 7000, "Vertical speed"),
    ("heading", "deg", "FLIGHT", 0, 360, 0, 360, "Aircraft heading"),
    ("pitch", "deg", "FLIGHT", -8, 18, -15, 25, "Pitch attitude"),
    ("roll", "deg", "FLIGHT", -35, 35, -55, 55, "Roll attitude"),
    ("yaw", "deg", "FLIGHT", -10, 10, -20, 20, "Yaw angle"),
    ("angle_of_attack", "deg", "FLIGHT", -2, 18, -5, 24, "Angle of attack"),
    ("latitude", "deg", "FLIGHT", -90, 90, -90, 90, "Latitude"),
    ("longitude", "deg", "FLIGHT", -180, 180, -180, 180, "Longitude"),
    ("weight", "kg", "FLIGHT", 25000, 350000, 20000, 380000, "Estimated aircraft gross weight"),
    ("cg_position", "pct_mac", "FLIGHT", 15, 40, 10, 45, "Center of gravity position"),
    ("flap_position", "deg", "FLIGHT", 0, 40, 0, 45, "Trailing edge flap position"),
    ("slat_position", "deg", "FLIGHT", 0, 25, 0, 30, "Leading edge slat position"),
    ("spoiler_position", "pct", "FLIGHT", 0, 100, 0, 100, "Spoiler deployment percentage"),
    ("gear_position", "pct", "FLIGHT", 0, 100, 0, 100, "Landing gear extension percentage"),
    ("autopilot_engaged", "flag", "FLIGHT", 0, 1, 0, 1, "Autopilot engagement flag"),
    ("throttle_position", "pct", "FLIGHT", 0, 100, 0, 100, "Throttle lever angle"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in flight_params)

hydraulic_params = [
    ("hyd_press_sys1", "psi", "HYDRAULIC", 2600, 3200, 500, 3600, "Hydraulic pressure system 1"),
    ("hyd_press_sys2", "psi", "HYDRAULIC", 2600, 3200, 500, 3600, "Hydraulic pressure system 2"),
    ("hyd_press_sys3", "psi", "HYDRAULIC", 2600, 3200, 500, 3600, "Hydraulic pressure system 3"),
    ("hyd_temp_sys1", "degC", "HYDRAULIC", 35, 85, 15, 110, "Hydraulic temperature system 1"),
    ("hyd_temp_sys2", "degC", "HYDRAULIC", 35, 85, 15, 110, "Hydraulic temperature system 2"),
    ("hyd_temp_sys3", "degC", "HYDRAULIC", 35, 85, 15, 110, "Hydraulic temperature system 3"),
    ("hyd_fluid_level_1", "pct", "HYDRAULIC", 70, 100, 10, 100, "Hydraulic fluid level reservoir 1"),
    ("hyd_fluid_level_2", "pct", "HYDRAULIC", 70, 100, 10, 100, "Hydraulic fluid level reservoir 2"),
    ("hyd_fluid_level_3", "pct", "HYDRAULIC", 70, 100, 10, 100, "Hydraulic fluid level reservoir 3"),
    ("hyd_pump_status", "flag", "HYDRAULIC", 0, 1, 0, 1, "Hydraulic pump operating status"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in hydraulic_params)

electrical_params = [
    ("gen_voltage_1", "volts", "ELECTRICAL", 110, 118, 80, 130, "Generator 1 output voltage"),
    ("gen_voltage_2", "volts", "ELECTRICAL", 110, 118, 80, 130, "Generator 2 output voltage"),
    ("gen_freq_1", "hz", "ELECTRICAL", 395, 405, 360, 430, "Generator 1 frequency"),
    ("gen_freq_2", "hz", "ELECTRICAL", 395, 405, 360, 430, "Generator 2 frequency"),
    ("bus_voltage_ac", "volts", "ELECTRICAL", 110, 120, 80, 130, "Main AC bus voltage"),
    ("bus_voltage_dc", "volts", "ELECTRICAL", 26, 29, 18, 32, "Main DC bus voltage"),
    ("battery_voltage", "volts", "ELECTRICAL", 24, 28, 18, 30, "Battery voltage"),
    ("battery_temp", "degC", "ELECTRICAL", 15, 55, -5, 85, "Battery temperature"),
    ("battery_charge", "pct", "ELECTRICAL", 65, 100, 10, 100, "Battery charge level"),
    ("apu_gen_voltage", "volts", "ELECTRICAL", 0, 118, 0, 130, "APU generator voltage"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in electrical_params)

environmental_params = [
    ("oat", "degC", "ENVIRONMENTAL", -65, 40, -80, 50, "Outside air temperature"),
    ("tat", "degC", "ENVIRONMENTAL", -55, 60, -70, 80, "Total air temperature"),
    ("wind_speed", "kts", "ENVIRONMENTAL", 0, 160, 0, 220, "Wind speed"),
    ("wind_direction", "deg", "ENVIRONMENTAL", 0, 360, 0, 360, "Wind direction"),
    ("baro_pressure", "hpa", "ENVIRONMENTAL", 200, 1050, 150, 1100, "Barometric pressure"),
    ("humidity", "pct", "ENVIRONMENTAL", 5, 100, 0, 100, "Relative humidity"),
    ("icing_indicator", "flag", "ENVIRONMENTAL", 0, 1, 0, 1, "Icing likelihood flag"),
    ("turbulence_index", "index", "ENVIRONMENTAL", 0, 1, 0, 1.5, "Derived turbulence intensity"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in environmental_params)

pneumatic_ecs_params = [
    ("pack_flow_1", "pct", "PNEUMATIC_ECS", 35, 100, 0, 120, "Left air conditioning pack flow"),
    ("pack_flow_2", "pct", "PNEUMATIC_ECS", 35, 100, 0, 120, "Right air conditioning pack flow"),
    ("cabin_pressure", "psi", "PNEUMATIC_ECS", 9.5, 14.9, 7.0, 16.0, "Cabin absolute pressure"),
    ("cabin_temp", "degC", "PNEUMATIC_ECS", 18, 28, 10, 35, "Cabin temperature"),
    ("cabin_altitude", "ft", "PNEUMATIC_ECS", 0, 8500, 0, 12000, "Cabin altitude"),
    ("diff_pressure", "psi", "PNEUMATIC_ECS", 0, 8.8, 0, 10.5, "Pressurization differential pressure"),
    ("outflow_valve_pos", "pct", "PNEUMATIC_ECS", 5, 90, 0, 100, "Outflow valve position"),
    ("pressurization_rate", "fpm", "PNEUMATIC_ECS", -500, 500, -1200, 1200, "Cabin pressurization rate"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in pneumatic_ecs_params)

fuel_params = [
    ("fuel_qty_total", "kg", "FUEL", 1000, 180000, 0, 200000, "Total fuel quantity"),
    ("fuel_qty_left", "kg", "FUEL", 500, 90000, 0, 100000, "Left tank fuel quantity"),
    ("fuel_qty_right", "kg", "FUEL", 500, 90000, 0, 100000, "Right tank fuel quantity"),
    ("fuel_qty_center", "kg", "FUEL", 0, 40000, 0, 60000, "Center tank fuel quantity"),
    ("fuel_temp", "degC", "FUEL", -45, 35, -55, 50, "Fuel temperature"),
    ("fuel_pressure", "psi", "FUEL", 18, 42, 5, 55, "Fuel line pressure"),
    ("fuel_imbalance", "kg", "FUEL", 0, 450, 0, 4000, "Absolute fuel imbalance between wing tanks"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in fuel_params)

landing_gear_params = [
    ("brake_temp_l1", "degC", "LANDING_GEAR", 20, 420, 10, 900, "Brake temperature left wheel 1"),
    ("brake_temp_l2", "degC", "LANDING_GEAR", 20, 420, 10, 900, "Brake temperature left wheel 2"),
    ("brake_temp_r1", "degC", "LANDING_GEAR", 20, 420, 10, 900, "Brake temperature right wheel 1"),
    ("brake_temp_r2", "degC", "LANDING_GEAR", 20, 420, 10, 900, "Brake temperature right wheel 2"),
    ("tire_press_nose", "psi", "LANDING_GEAR", 150, 215, 110, 235, "Nose tire pressure"),
    ("tire_press_left", "psi", "LANDING_GEAR", 165, 220, 120, 240, "Left main tire pressure"),
    ("tire_press_right", "psi", "LANDING_GEAR", 165, 220, 120, 240, "Right main tire pressure"),
    ("gear_actuator_press", "psi", "LANDING_GEAR", 2400, 3200, 500, 3600, "Landing gear actuator pressure"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in landing_gear_params)

flight_control_params = [
    ("aileron_pos_l", "deg", "FLIGHT_CONTROLS", -25, 25, -35, 35, "Left aileron position"),
    ("aileron_pos_r", "deg", "FLIGHT_CONTROLS", -25, 25, -35, 35, "Right aileron position"),
    ("elevator_pos", "deg", "FLIGHT_CONTROLS", -25, 25, -35, 35, "Elevator position"),
    ("rudder_pos", "deg", "FLIGHT_CONTROLS", -30, 30, -40, 40, "Rudder position"),
    ("trim_pos", "deg", "FLIGHT_CONTROLS", -12, 12, -18, 18, "Stabilizer trim position"),
    ("stick_force_pitch", "newtons", "FLIGHT_CONTROLS", 0, 250, 0, 420, "Pitch axis stick force"),
    ("stick_force_roll", "newtons", "FLIGHT_CONTROLS", 0, 220, 0, 380, "Roll axis stick force"),
    ("rudder_pedal_force", "newtons", "FLIGHT_CONTROLS", 0, 200, 0, 320, "Rudder pedal force"),
]
PARAMETER_REGISTRY.extend(make_param(*spec) for spec in flight_control_params)

PARAMETER_LOOKUP = {item["param_name"]: item for item in PARAMETER_REGISTRY}
PARAMETER_NAMES = [item["param_name"] for item in PARAMETER_REGISTRY]
METADATA_COLUMNS = [
    "flight_id",
    "airline",
    "aircraft_type",
    "tail_number",
    "origin",
    "destination",
    "timestamp",
    "event_date",
    "flight_phase",
    "record_sequence",
    "anomaly_type",
    "anomaly_severity",
]

registry_summary_pdf = (
    pd.DataFrame(PARAMETER_REGISTRY)
    .groupby("system_group", as_index=False)
    .agg(parameter_count=("param_name", "count"))
)

display(registry_summary_pdf)
print(f"Registered {len(PARAMETER_NAMES)} sensor parameters and {len(METADATA_COLUMNS)} metadata columns.")


system_group,parameter_count
ELECTRICAL,10
ENGINE,30
ENVIRONMENTAL,8
FLIGHT,20
FLIGHT_CONTROLS,8
FUEL,7
HYDRAULIC,10
LANDING_GEAR,8
PNEUMATIC_ECS,8


Registered 109 sensor parameters and 12 metadata columns.


In [0]:
PHASES = [
    "TAXI_OUT",
    "TAKEOFF",
    "CLIMB",
    "CRUISE",
    "DESCENT",
    "APPROACH_LANDING",
]

PHASE_DURATION_RATIOS = np.array([0.05, 0.03, 0.15, 0.55, 0.15, 0.07], dtype=float)
PHASE_DURATION_RATIOS = PHASE_DURATION_RATIOS / PHASE_DURATION_RATIOS.sum()
PHASE_CUMULATIVE = np.insert(np.cumsum(PHASE_DURATION_RATIOS), 0, 0.0)


def smooth_series(values: np.ndarray, window: int = 9) -> np.ndarray:
    if window <= 1:
        return values
    kernel = np.ones(window, dtype=float) / window
    padded = np.pad(values, (window // 2, window // 2), mode="edge")
    return np.convolve(padded, kernel, mode="valid")


def phase_vector(n_records: int) -> np.ndarray:
    phase_counts = np.round(PHASE_DURATION_RATIOS * n_records).astype(int)
    phase_counts[-1] += n_records - int(phase_counts.sum())
    values = np.repeat(PHASES, phase_counts)
    return values[:n_records]


def interpolate_profile(anchor_values: List[float], n_records: int, smooth_window: int = 11) -> np.ndarray:
    if len(anchor_values) != len(PHASES) + 1:
        raise ValueError("anchor_values must contain one more entry than the number of phases")
    progress = np.linspace(0.0, 1.0, n_records)
    series = np.interp(progress, PHASE_CUMULATIVE, anchor_values)
    return smooth_series(series, smooth_window)


def estimate_block_hours(route: Dict[str, str]) -> float:
    distance_nm = great_circle_nm(route["origin"], route["destination"])
    return float(np.clip(0.45 + (distance_nm / 430.0), 1.0, 13.5))


def build_flight_profile(route: Dict[str, str], n_records: int, rng: np.random.Generator) -> Dict[str, np.ndarray]:
    origin = AIRPORTS[route["origin"]]
    destination = AIRPORTS[route["destination"]]
    distance_nm = great_circle_nm(route["origin"], route["destination"])
    block_hours = estimate_block_hours(route)
    record_interval_seconds = float(np.clip(block_hours * 3600 / n_records, 5.0, 12.0))
    offsets = np.arange(n_records, dtype=float) * record_interval_seconds
    progress = np.linspace(0.0, 1.0, n_records)

    cruise_altitude = float(np.clip(24000 + distance_nm * 9, 28000, 41000))
    cruise_ias = float(np.clip(280 + distance_nm * 0.03, 290, 340))
    cruise_ground_speed = cruise_ias + rng.uniform(25, 60)
    cruise_mach = float(np.clip(0.70 + distance_nm / 12000, 0.74, 0.86))
    starting_weight = float(np.clip(42000 + distance_nm * rng.uniform(18, 45), 38000, 260000))
    fuel_burn_total = starting_weight * rng.uniform(0.04, 0.12)

    altitude = interpolate_profile([0, 0, 1800, cruise_altitude * 0.68, cruise_altitude, cruise_altitude * 0.35, 0], n_records)
    indicated_airspeed = interpolate_profile([12, 35, 185, 285, cruise_ias, 250, 135], n_records)
    ground_speed = interpolate_profile([10, 28, 170, 315, cruise_ground_speed, 255, 18], n_records)
    mach_number = interpolate_profile([0.0, 0.0, 0.22, 0.56, cruise_mach, 0.46, 0.08], n_records)
    vertical_speed = interpolate_profile([0, 0, 2100, 1300, 0, -1650, -450], n_records, smooth_window=7)
    heading = (np.degrees(np.arctan2(destination["lon"] - origin["lon"], destination["lat"] - origin["lat"])) + 360.0) % 360.0
    heading_series = (heading + rng.normal(0, 2.2, n_records)) % 360.0
    pitch = interpolate_profile([0, 1.5, 10.0, 5.5, 2.0, -2.5, 1.0], n_records)
    roll = rng.normal(0, 2.2, n_records) + np.sin(progress * 16 * np.pi) * 1.4
    yaw = rng.normal(0, 0.8, n_records)
    angle_of_attack = interpolate_profile([4.0, 5.5, 10.5, 6.5, 4.0, 7.0, 8.5], n_records)
    flap_position = interpolate_profile([8, 10, 15, 5, 0, 12, 25], n_records)
    slat_position = interpolate_profile([7, 10, 12, 4, 0, 10, 22], n_records)
    spoiler_position = np.clip(np.abs(rng.normal(0.0, 1.2, n_records)), 0, 12)
    spoiler_position[phase_vector(n_records) == "APPROACH_LANDING"] += np.linspace(5, 20, np.sum(phase_vector(n_records) == "APPROACH_LANDING"))
    gear_position = interpolate_profile([100, 100, 0, 0, 0, 100, 100], n_records)
    autopilot_engaged = np.where(np.isin(phase_vector(n_records), ["CLIMB", "CRUISE", "DESCENT"]), 1, 0)
    throttle_position = interpolate_profile([18, 24, 96, 84, 66, 36, 12], n_records)
    weight = starting_weight - np.linspace(0.0, fuel_burn_total, n_records)
    cg_position = 24 + np.sin(progress * np.pi) * 3 + rng.normal(0, 0.25, n_records)

    latitude = np.linspace(origin["lat"], destination["lat"], n_records) + np.sin(progress * np.pi) * rng.uniform(-1.5, 1.5)
    longitude = np.linspace(origin["lon"], destination["lon"], n_records) + np.sin(progress * np.pi) * rng.uniform(-2.5, 2.5)

    phase_values = phase_vector(n_records)
    return {
        "phase": phase_values,
        "offset_seconds": offsets,
        "progress": progress,
        "distance_nm": np.full(n_records, distance_nm),
        "record_interval_seconds": np.full(n_records, record_interval_seconds),
        "altitude": altitude,
        "indicated_airspeed": indicated_airspeed,
        "ground_speed": ground_speed,
        "mach_number": mach_number,
        "vertical_speed": vertical_speed,
        "heading": heading_series,
        "pitch": pitch,
        "roll": roll,
        "yaw": yaw,
        "angle_of_attack": angle_of_attack,
        "latitude": latitude,
        "longitude": longitude,
        "weight": weight,
        "cg_position": cg_position,
        "flap_position": flap_position,
        "slat_position": slat_position,
        "spoiler_position": spoiler_position,
        "gear_position": gear_position,
        "autopilot_engaged": autopilot_engaged,
        "throttle_position": throttle_position,
    }


demo_profile_rng = np.random.default_rng(RANDOM_SEED)
demo_profile = build_flight_profile(ROUTE_CATALOG[0], 120, demo_profile_rng)
demo_profile_pdf = pd.DataFrame(
    {
        "phase": demo_profile["phase"],
        "altitude": demo_profile["altitude"],
        "ias": demo_profile["indicated_airspeed"],
        "ground_speed": demo_profile["ground_speed"],
        "mach": demo_profile["mach_number"],
        "vertical_speed": demo_profile["vertical_speed"],
    }
)
display(demo_profile_pdf.head(12))


phase,altitude,ias,ground_speed,mach,vertical_speed
TAXI_OUT,0.0,17.271199388846448,14.125286478227656,0.0,0.0
TAXI_OUT,2.2918258212375777,19.55309396485867,15.942449707155589,2.8011204481792623E-4,0.0
TAXI_OUT,50.42016806722686,25.654698242933534,21.375604787369493,0.006162464985994395,0.0
TAXI_OUT,144.38502673796788,35.57601222307105,30.424751718869366,0.017647058823529405,4.201680672268893
TAXI_OUT,284.18640183346054,49.3170359052712,43.089890501655205,0.03473389355742296,92.43697478991591
TAXI_OUT,511.57830404889205,65.28877005347593,57.989814107461164,0.05556506238859178,264.70588235294105
TAKEOFF,871.7942449707152,81.76979882862236,73.62821492233257,0.07812783295136234,521.0084033613443
TAKEOFF,1364.8342245989302,98.4087089381207,89.7300738477209,0.10242220524573464,817.9351740696277
TAKEOFF,1990.698242933537,115.20550038197096,106.29539088362617,0.12844817927170868,1108.4593837535012
TAKEOFF,2749.386299974535,132.16017316017314,123.32416603004839,0.15620575502928444,1392.581032412965


In [0]:
ANOMALY_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
ANOMALY_SEVERITY_PROBABILITIES = [0.50, 0.25, 0.15, 0.10]
SEVERITY_SCALE = {
    "LOW": 0.35,
    "MEDIUM": 0.65,
    "HIGH": 0.95,
    "CRITICAL": 1.35,
}

ANOMALY_TYPES = {
    "ENGINE_DEGRADATION": ["egt_eng1", "egt_eng2", "n1_speed_eng1", "n1_speed_eng2", "n2_speed_eng1", "n2_speed_eng2", "vibration_n1_eng1", "vibration_n1_eng2"],
    "OIL_SYSTEM_FAULT": ["oil_pressure_eng1", "oil_pressure_eng2", "oil_temp_eng1", "oil_temp_eng2"],
    "HYDRAULIC_LEAK": ["hyd_press_sys1", "hyd_press_sys2", "hyd_press_sys3", "hyd_fluid_level_1", "hyd_fluid_level_2", "hyd_fluid_level_3"],
    "ELECTRICAL_FAULT": ["gen_voltage_1", "gen_voltage_2", "gen_freq_1", "gen_freq_2", "bus_voltage_ac", "bus_voltage_dc"],
    "BLEED_AIR_LEAK": ["bleed_pressure_eng1", "bleed_pressure_eng2", "bleed_temp_eng1", "bleed_temp_eng2", "pack_flow_1", "pack_flow_2", "cabin_pressure"],
    "FUEL_IMBALANCE": ["fuel_qty_left", "fuel_qty_right", "fuel_imbalance"],
    "BRAKE_OVERHEAT": ["brake_temp_l1", "brake_temp_l2", "brake_temp_r1", "brake_temp_r2"],
    "VIBRATION_ANOMALY": ["vibration_n1_eng1", "vibration_n2_eng1", "vibration_n1_eng2", "vibration_n2_eng2"],
    "SENSOR_DRIFT": ["oat", "tat", "fuel_pressure", "cabin_pressure", "battery_temp"],
    "COMPRESSOR_STALL": ["n2_speed_eng1", "n2_speed_eng2", "egt_eng1", "egt_eng2", "thrust_output_eng1", "thrust_output_eng2"],
}


def sample_anomaly_config(rng: np.random.Generator) -> Dict[str, str]:
    anomaly_type = rng.choice(list(ANOMALY_TYPES.keys()))
    severity = rng.choice(ANOMALY_SEVERITIES, p=ANOMALY_SEVERITY_PROBABILITIES)
    return {"anomaly_type": str(anomaly_type), "anomaly_severity": str(severity)}


def anomaly_window(n_records: int, rng: np.random.Generator, min_width: float = 0.10, max_width: float = 0.30) -> np.ndarray:
    start_ratio = float(rng.uniform(0.08, 0.82))
    width_ratio = float(rng.uniform(min_width, max_width))
    start_idx = int(start_ratio * n_records)
    end_idx = min(n_records, int((start_ratio + width_ratio) * n_records))
    mask = np.zeros(n_records, dtype=bool)
    mask[start_idx:end_idx] = True
    return mask


def clip_parameter(df: pd.DataFrame, column_name: str) -> None:
    bounds = PARAMETER_LOOKUP[column_name]
    df[column_name] = df[column_name].clip(bounds["anomaly_min"], bounds["anomaly_max"])


def apply_anomaly(df: pd.DataFrame, anomaly_config: Optional[Dict[str, str]], rng: np.random.Generator) -> pd.DataFrame:
    if anomaly_config is None:
        df["anomaly_type"] = None
        df["anomaly_severity"] = None
        return df

    anomaly_type = anomaly_config["anomaly_type"]
    anomaly_severity = anomaly_config["anomaly_severity"]
    scale = SEVERITY_SCALE[anomaly_severity]
    mask = anomaly_window(len(df), rng)
    ramp = np.linspace(0.0, 1.0, max(mask.sum(), 1))
    landing_mask = (df["flight_phase"] == "APPROACH_LANDING").to_numpy()
    selected_engine = int(rng.choice([1, 2]))
    selected_hyd_system = int(rng.choice([1, 2, 3]))

    if anomaly_type == "ENGINE_DEGRADATION":
        for engine_id in (1, 2):
            df.loc[mask, f"egt_eng{engine_id}"] += ramp * 110 * scale
            df.loc[mask, f"n1_speed_eng{engine_id}"] -= ramp * 12 * scale
            df.loc[mask, f"n2_speed_eng{engine_id}"] -= ramp * 8 * scale
            df.loc[mask, f"vibration_n1_eng{engine_id}"] += ramp * 1.1 * scale
            df.loc[mask, f"vibration_n2_eng{engine_id}"] += ramp * 0.8 * scale
    elif anomaly_type == "OIL_SYSTEM_FAULT":
        df.loc[mask, f"oil_pressure_eng{selected_engine}"] -= (25 + 18 * scale) * np.maximum(ramp, 0.3)
        df.loc[mask, f"oil_temp_eng{selected_engine}"] += (18 + 35 * scale) * np.maximum(ramp, 0.2)
    elif anomaly_type == "HYDRAULIC_LEAK":
        df.loc[mask, f"hyd_press_sys{selected_hyd_system}"] -= (500 + 900 * scale) * np.maximum(ramp, 0.2)
        df.loc[mask, f"hyd_fluid_level_{selected_hyd_system}"] -= (8 + 32 * scale) * np.maximum(ramp, 0.3)
        df.loc[mask, "gear_actuator_press"] -= (200 + 400 * scale) * np.maximum(ramp, 0.2)
    elif anomaly_type == "ELECTRICAL_FAULT":
        swing = rng.normal(0, 4 + 8 * scale, mask.sum())
        for col in ["gen_voltage_1", "gen_voltage_2", "bus_voltage_ac"]:
            df.loc[mask, col] += swing
        for col in ["gen_freq_1", "gen_freq_2"]:
            df.loc[mask, col] += rng.normal(0, 5 + 10 * scale, mask.sum())
        df.loc[mask, "bus_voltage_dc"] += rng.normal(-2 * scale, 1.5 + 1.2 * scale, mask.sum())
    elif anomaly_type == "BLEED_AIR_LEAK":
        for engine_id in (1, 2):
            df.loc[mask, f"bleed_pressure_eng{engine_id}"] -= (5 + 10 * scale) * np.maximum(ramp, 0.2)
            df.loc[mask, f"bleed_temp_eng{engine_id}"] += (12 + 24 * scale) * np.maximum(ramp, 0.2)
        for col in ["pack_flow_1", "pack_flow_2"]:
            df.loc[mask, col] -= (5 + 16 * scale) * np.maximum(ramp, 0.2)
        df.loc[mask, "cabin_pressure"] -= 0.4 * scale * np.maximum(ramp, 0.2)
        df.loc[mask, "cabin_altitude"] += (400 + 1000 * scale) * np.maximum(ramp, 0.2)
    elif anomaly_type == "FUEL_IMBALANCE":
        imbalance_shift = (200 + 1000 * scale) * np.maximum(ramp, 0.2)
        df.loc[mask, "fuel_qty_left"] += imbalance_shift
        df.loc[mask, "fuel_qty_right"] -= imbalance_shift
        df.loc[mask, "fuel_imbalance"] += np.abs(imbalance_shift) * 2
    elif anomaly_type == "BRAKE_OVERHEAT":
        overheat_mask = landing_mask.copy()
        if not overheat_mask.any():
            overheat_mask[-int(len(df) * 0.08):] = True
        for col in ["brake_temp_l1", "brake_temp_l2", "brake_temp_r1", "brake_temp_r2"]:
            df.loc[overheat_mask, col] += np.linspace(140, 420 * scale, overheat_mask.sum())
    elif anomaly_type == "VIBRATION_ANOMALY":
        for col in [f"vibration_n1_eng{selected_engine}", f"vibration_n2_eng{selected_engine}"]:
            df.loc[mask, col] += rng.normal(0.9 * scale, 0.25, mask.sum()) + np.maximum(ramp, 0.2)
    elif anomaly_type == "SENSOR_DRIFT":
        drift = np.linspace(0, scale, len(df))
        df["oat"] += drift * 8
        df["tat"] += drift * 7
        df["fuel_pressure"] -= drift * 3
        df["cabin_pressure"] += drift * 0.7
        df["battery_temp"] += drift * 6
    elif anomaly_type == "COMPRESSOR_STALL":
        stall_mask = anomaly_window(len(df), rng, min_width=0.02, max_width=0.06)
        df.loc[stall_mask, f"n2_speed_eng{selected_engine}"] -= (12 + 16 * scale)
        df.loc[stall_mask, f"egt_eng{selected_engine}"] += (85 + 120 * scale)
        df.loc[stall_mask, f"thrust_output_eng{selected_engine}"] -= (10 + 22 * scale)
        df.loc[stall_mask, f"vibration_n2_eng{selected_engine}"] += (0.5 + 0.8 * scale)

    touched_columns = set(ANOMALY_TYPES[anomaly_type])
    if anomaly_type == "HYDRAULIC_LEAK":
        touched_columns.add("gear_actuator_press")
    if anomaly_type == "BLEED_AIR_LEAK":
        touched_columns.update(["cabin_pressure", "cabin_altitude"])
    if anomaly_type == "FUEL_IMBALANCE":
        touched_columns.update(["fuel_qty_left", "fuel_qty_right", "fuel_imbalance"])
    if anomaly_type == "BRAKE_OVERHEAT":
        touched_columns.update(["brake_temp_l1", "brake_temp_l2", "brake_temp_r1", "brake_temp_r2"])
    if anomaly_type == "SENSOR_DRIFT":
        touched_columns.update(["oat", "tat", "fuel_pressure", "cabin_pressure", "battery_temp"])

    for column_name in touched_columns:
        if column_name in df.columns:
            clip_parameter(df, column_name)

    df["anomaly_type"] = anomaly_type
    df["anomaly_severity"] = anomaly_severity
    return df


rng_anomaly = np.random.default_rng(RANDOM_SEED)
display(pd.DataFrame([sample_anomaly_config(rng_anomaly) for _ in range(5)]))


anomaly_type,anomaly_severity
ENGINE_DEGRADATION,LOW
VIBRATION_ANOMALY,HIGH
ENGINE_DEGRADATION,LOW
BRAKE_OVERHEAT,CRITICAL
VIBRATION_ANOMALY,HIGH


In [0]:
def finalize_parameter_bounds(df: pd.DataFrame) -> pd.DataFrame:
    for column_name in PARAMETER_NAMES:
        clip_parameter(df, column_name)
    return df


def generate_flight_data(
    flight_id: str,
    airline: str,
    aircraft: Dict[str, Any],
    route: Dict[str, str],
    anomaly_config: Optional[Dict[str, str]] = None,
    departure_time: Optional[datetime] = None,
    n_records: int = RECORDS_PER_FLIGHT,
    random_state: Optional[int] = None,
) -> pd.DataFrame:
    seed_value = random_state if random_state is not None else (abs(hash((flight_id, aircraft["tail_number"], RANDOM_SEED))) % (2**32))
    rng = np.random.default_rng(seed_value)

    if departure_time is None:
        departure_time = utc_now_naive() - timedelta(minutes=int(rng.integers(0, 365 * 24 * 60)))

    profile = build_flight_profile(route, n_records, rng)
    phase = profile["phase"]
    progress = profile["progress"]
    altitude = profile["altitude"]
    indicated_airspeed = profile["indicated_airspeed"]
    ground_speed = profile["ground_speed"]
    mach_number = profile["mach_number"]
    vertical_speed = profile["vertical_speed"]
    heading = profile["heading"]
    pitch = profile["pitch"]
    roll = profile["roll"]
    yaw = profile["yaw"]
    angle_of_attack = profile["angle_of_attack"]
    throttle_position = np.clip(profile["throttle_position"], 0, 100)

    timestamps = pd.to_datetime(departure_time) + pd.to_timedelta(profile["offset_seconds"], unit="s")
    shared_noise = rng.normal(0, 1, n_records)
    environment_noise = rng.normal(0, 1, n_records)
    throttle_norm = throttle_position / 100.0
    altitude_norm = altitude / max(float(np.nanmax(altitude)), 1.0)
    landing_mask = phase == "APPROACH_LANDING"
    cruise_mask = phase == "CRUISE"

    data: Dict[str, Any] = {
        "flight_id": flight_id,
        "airline": airline,
        "aircraft_type": aircraft["aircraft_type"],
        "tail_number": aircraft["tail_number"],
        "origin": route["origin"],
        "destination": route["destination"],
        "timestamp": timestamps,
        "event_date": pd.to_datetime(timestamps).date,
        "flight_phase": phase,
        "record_sequence": np.arange(1, n_records + 1, dtype=int),
        "anomaly_type": None,
        "anomaly_severity": None,
        "altitude": altitude + rng.normal(0, 45, n_records),
        "indicated_airspeed": indicated_airspeed + rng.normal(0, 2.8, n_records),
        "ground_speed": ground_speed + rng.normal(0, 3.6, n_records),
        "mach_number": np.clip(mach_number + rng.normal(0, 0.004, n_records), 0, 0.95),
        "vertical_speed": vertical_speed + rng.normal(0, 80, n_records),
        "heading": heading,
        "pitch": pitch + rng.normal(0, 0.25, n_records),
        "roll": roll + rng.normal(0, 0.3, n_records),
        "yaw": yaw + rng.normal(0, 0.15, n_records),
        "angle_of_attack": angle_of_attack + rng.normal(0, 0.2, n_records),
        "latitude": profile["latitude"] + rng.normal(0, 0.01, n_records),
        "longitude": profile["longitude"] + rng.normal(0, 0.01, n_records),
        "weight": profile["weight"] + rng.normal(0, 35, n_records),
        "cg_position": profile["cg_position"],
        "flap_position": np.clip(profile["flap_position"] + rng.normal(0, 0.25, n_records), 0, 45),
        "slat_position": np.clip(profile["slat_position"] + rng.normal(0, 0.25, n_records), 0, 30),
        "spoiler_position": np.clip(profile["spoiler_position"] + rng.normal(0, 0.4, n_records), 0, 100),
        "gear_position": np.clip(profile["gear_position"] + rng.normal(0, 0.2, n_records), 0, 100),
        "autopilot_engaged": profile["autopilot_engaged"].astype(int),
        "throttle_position": throttle_position,
    }

    outside_air_temp = rng.uniform(-6, 32) - altitude / 1000.0 * 1.98 + environment_noise * 0.8
    data["oat"] = np.clip(outside_air_temp, -80, 50)
    data["tat"] = np.clip(outside_air_temp + mach_number * 35 + rng.normal(0, 0.8, n_records), -70, 80)
    data["wind_speed"] = np.clip(12 + altitude_norm * 95 + rng.normal(0, 8, n_records), 0, 220)
    data["wind_direction"] = np.mod(heading + rng.normal(25, 35, n_records), 360)
    data["baro_pressure"] = np.clip(1013.25 * np.exp(-altitude / 26000.0) + rng.normal(0, 2.5, n_records), 150, 1100)
    data["humidity"] = np.clip(70 - altitude_norm * 45 + rng.normal(0, 6, n_records), 0, 100)
    icing_probability = ((data["oat"] < 5) & (data["oat"] > -20) & (data["humidity"] > 60)).astype(int)
    data["icing_indicator"] = icing_probability
    data["turbulence_index"] = np.clip(np.abs(rng.normal(0.18 + altitude_norm * 0.12, 0.08, n_records)), 0, 1.5)

    start_fuel = float(np.clip(data["weight"][0] * 0.22, 6000, 90000))
    fuel_remaining = np.clip(start_fuel - np.linspace(0, start_fuel * rng.uniform(0.45, 0.82), n_records), 1400, None)
    center_ratio = np.clip(0.32 - progress * 0.36, 0, 0.28)
    fuel_center = np.clip(fuel_remaining * center_ratio, 0, fuel_remaining * 0.35)
    wing_total = np.maximum(fuel_remaining - fuel_center, 0)
    wing_split_bias = rng.normal(0, 35, n_records)
    data["fuel_qty_left"] = np.clip(wing_total / 2 + wing_split_bias, 0, None)
    data["fuel_qty_right"] = np.clip(wing_total / 2 - wing_split_bias, 0, None)
    data["fuel_qty_center"] = fuel_center
    data["fuel_qty_total"] = data["fuel_qty_left"] + data["fuel_qty_right"] + fuel_center
    data["fuel_temp"] = np.clip(data["oat"] + rng.normal(5, 1.2, n_records), -55, 50)
    data["fuel_pressure"] = np.clip(22 + throttle_norm * 16 + rng.normal(0, 1.0, n_records), 5, 55)
    data["fuel_imbalance"] = np.abs(data["fuel_qty_left"] - data["fuel_qty_right"])

    control_activity = np.abs(data["roll"]) / 35 + np.abs(data["pitch"]) / 18
    data["hyd_press_sys1"] = np.clip(3000 - control_activity * 120 + rng.normal(0, 25, n_records), 500, 3600)
    data["hyd_press_sys2"] = np.clip(2985 - control_activity * 110 + rng.normal(0, 25, n_records), 500, 3600)
    data["hyd_press_sys3"] = np.clip(3020 - control_activity * 105 + rng.normal(0, 25, n_records), 500, 3600)
    data["hyd_temp_sys1"] = np.clip(44 + throttle_norm * 18 + rng.normal(0, 1.8, n_records), 15, 110)
    data["hyd_temp_sys2"] = np.clip(45 + throttle_norm * 17 + rng.normal(0, 1.8, n_records), 15, 110)
    data["hyd_temp_sys3"] = np.clip(46 + throttle_norm * 16 + rng.normal(0, 1.8, n_records), 15, 110)
    data["hyd_fluid_level_1"] = np.clip(94 - progress * 3 + rng.normal(0, 0.3, n_records), 10, 100)
    data["hyd_fluid_level_2"] = np.clip(93 - progress * 3 + rng.normal(0, 0.3, n_records), 10, 100)
    data["hyd_fluid_level_3"] = np.clip(95 - progress * 3 + rng.normal(0, 0.3, n_records), 10, 100)
    data["hyd_pump_status"] = np.ones(n_records, dtype=int)

    data["gen_voltage_1"] = np.clip(115 + rng.normal(0, 0.9, n_records), 80, 130)
    data["gen_voltage_2"] = np.clip(115 + rng.normal(0, 0.9, n_records), 80, 130)
    data["gen_freq_1"] = np.clip(400 + rng.normal(0, 1.2, n_records), 360, 430)
    data["gen_freq_2"] = np.clip(400 + rng.normal(0, 1.2, n_records), 360, 430)
    data["bus_voltage_ac"] = np.clip(115 + rng.normal(0, 0.7, n_records), 80, 130)
    data["bus_voltage_dc"] = np.clip(28 + rng.normal(0, 0.25, n_records), 18, 32)
    data["battery_voltage"] = np.clip(26.2 + rng.normal(0, 0.18, n_records), 18, 30)
    data["battery_temp"] = np.clip(22 + throttle_norm * 8 + rng.normal(0, 0.9, n_records), -5, 85)
    data["battery_charge"] = np.clip(96 - progress * 6 + rng.normal(0, 0.3, n_records), 10, 100)
    data["apu_gen_voltage"] = np.where(phase == "TAXI_OUT", 115 + rng.normal(0, 1.0, n_records), rng.normal(0, 0.05, n_records))

    pressurization_altitude = np.clip(altitude * 0.22 + 400, 0, 8500)
    data["pack_flow_1"] = np.clip(74 - cruise_mask.astype(int) * 10 + rng.normal(0, 2.5, n_records), 0, 120)
    data["pack_flow_2"] = np.clip(73 - cruise_mask.astype(int) * 10 + rng.normal(0, 2.5, n_records), 0, 120)
    data["cabin_altitude"] = pressurization_altitude
    data["diff_pressure"] = np.clip((altitude - pressurization_altitude) / 5500, 0, 8.8)
    data["cabin_pressure"] = np.clip(14.7 - data["diff_pressure"] + rng.normal(0, 0.05, n_records), 7.0, 16.0)
    data["cabin_temp"] = np.clip(22 + rng.normal(0, 0.6, n_records), 10, 35)
    data["outflow_valve_pos"] = np.clip(58 - altitude_norm * 28 + rng.normal(0, 2, n_records), 0, 100)
    data["pressurization_rate"] = np.clip(vertical_speed * 0.12 + rng.normal(0, 18, n_records), -1200, 1200)

    brake_base = 65 + np.maximum(0, 1 - altitude_norm) * 22 + rng.normal(0, 3.0, n_records)
    brake_rise = np.zeros(n_records)
    if landing_mask.any():
        brake_rise[landing_mask] = np.linspace(60, 240, landing_mask.sum())
    data["brake_temp_l1"] = np.clip(brake_base + brake_rise + rng.normal(0, 4, n_records), 10, 900)
    data["brake_temp_l2"] = np.clip(brake_base + brake_rise + rng.normal(0, 4, n_records), 10, 900)
    data["brake_temp_r1"] = np.clip(brake_base + brake_rise + rng.normal(0, 4, n_records), 10, 900)
    data["brake_temp_r2"] = np.clip(brake_base + brake_rise + rng.normal(0, 4, n_records), 10, 900)
    data["tire_press_nose"] = np.clip(188 + rng.normal(0, 1.8, n_records), 110, 235)
    data["tire_press_left"] = np.clip(196 + rng.normal(0, 1.9, n_records), 120, 240)
    data["tire_press_right"] = np.clip(196 + rng.normal(0, 1.9, n_records), 120, 240)
    data["gear_actuator_press"] = np.clip(2950 - (data["gear_position"] / 100) * 120 + rng.normal(0, 30, n_records), 500, 3600)

    data["aileron_pos_l"] = np.clip(-data["roll"] * 0.55 + rng.normal(0, 0.5, n_records), -35, 35)
    data["aileron_pos_r"] = np.clip(data["roll"] * 0.55 + rng.normal(0, 0.5, n_records), -35, 35)
    data["elevator_pos"] = np.clip(data["pitch"] * 0.9 + rng.normal(0, 0.45, n_records), -35, 35)
    data["rudder_pos"] = np.clip(data["yaw"] * 1.8 + rng.normal(0, 0.35, n_records), -40, 40)
    data["trim_pos"] = np.clip(-data["cg_position"] * 0.08 + rng.normal(0, 0.25, n_records), -18, 18)
    data["stick_force_pitch"] = np.clip(np.abs(data["pitch"]) * 9 + rng.normal(12, 8, n_records), 0, 420)
    data["stick_force_roll"] = np.clip(np.abs(data["roll"]) * 6 + rng.normal(12, 7, n_records), 0, 380)
    data["rudder_pedal_force"] = np.clip(np.abs(data["yaw"]) * 14 + rng.normal(10, 6, n_records), 0, 320)

    for engine_id in (1, 2):
        engine_bias = 1.0 + rng.normal(0, 0.018)
        engine_noise = rng.normal(0, 1, n_records)
        thrust_output = np.clip(throttle_position * engine_bias + engine_noise * 0.8, 0, 110)
        data[f"thrust_output_eng{engine_id}"] = thrust_output
        data[f"egt_eng{engine_id}"] = np.clip(410 + thrust_output * 4.4 + altitude_norm * 30 + engine_noise * 4.5, 320, 1050)
        data[f"n1_speed_eng{engine_id}"] = np.clip(18 + thrust_output * 0.82 + engine_noise * 0.5, 10, 108)
        data[f"n2_speed_eng{engine_id}"] = np.clip(56 + thrust_output * 0.42 + engine_noise * 0.4, 40, 112)
        data[f"oil_pressure_eng{engine_id}"] = np.clip(34 + thrust_output * 0.28 - altitude_norm * 3 + engine_noise * 0.9, 5, 95)
        data[f"oil_temp_eng{engine_id}"] = np.clip(72 + thrust_output * 0.42 + engine_noise * 1.8, 20, 190)
        data[f"fuel_flow_eng{engine_id}"] = np.clip(280 + thrust_output * (42 + engine_id) + engine_noise * 20, 100, 7500)
        data[f"vibration_n1_eng{engine_id}"] = np.clip(0.08 + throttle_norm * 0.35 + np.abs(engine_noise) * 0.08, 0, 4)
        data[f"vibration_n2_eng{engine_id}"] = np.clip(0.06 + throttle_norm * 0.28 + np.abs(engine_noise) * 0.06, 0, 3.5)
        data[f"epr_eng{engine_id}"] = np.clip(1.02 + throttle_norm * 1.02 + engine_noise * 0.03, 0.8, 2.5)
        data[f"ff_ratio_eng{engine_id}"] = np.clip(0.92 + rng.normal(0, 0.03, n_records), 0.4, 1.8)
        data[f"bleed_pressure_eng{engine_id}"] = np.clip(22 + throttle_norm * 22 + engine_noise * 1.4, 5, 70)
        data[f"bleed_temp_eng{engine_id}"] = np.clip(135 + thrust_output * 1.05 + engine_noise * 4.5, 60, 340)
        data[f"starter_valve_pos_eng{engine_id}"] = np.where(np.arange(n_records) < 10, 100, 0)
        reverser_profile = np.zeros(n_records)
        if landing_mask.any():
            reverser_profile[landing_mask] = np.linspace(0, 82, landing_mask.sum())
        data[f"reverser_pos_eng{engine_id}"] = np.clip(reverser_profile + rng.normal(0, 1.5, n_records), 0, 100)

    flight_pdf = pd.DataFrame(data)
    flight_pdf = apply_anomaly(flight_pdf, anomaly_config, rng)
    flight_pdf = finalize_parameter_bounds(flight_pdf)
    flight_pdf = flight_pdf.sort_values("timestamp").reset_index(drop=True)

    missing_columns = [column_name for column_name in (METADATA_COLUMNS + PARAMETER_NAMES) if column_name not in flight_pdf.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in generated flight data: {missing_columns}")

    return flight_pdf[METADATA_COLUMNS + PARAMETER_NAMES]


sample_airline = AIRLINES[0]
sample_aircraft = AIRLINE_FLEET_MAP[sample_airline][0]
sample_route = AIRLINE_ROUTE_MAP[sample_airline][0]
sample_flight_pdf = generate_flight_data(
    flight_id="FLT_DEMO_0001",
    airline=sample_airline,
    aircraft=sample_aircraft,
    route=sample_route,
    anomaly_config=sample_anomaly_config(np.random.default_rng(RANDOM_SEED + 100)),
    n_records=25,
    random_state=RANDOM_SEED + 101,
)
display(sample_flight_pdf.head(10))
print(sample_flight_pdf[["flight_id", "flight_phase", "anomaly_type", "anomaly_severity"]].drop_duplicates().head(6))


flight_id,airline,aircraft_type,tail_number,origin,destination,timestamp,event_date,flight_phase,record_sequence,anomaly_type,anomaly_severity,egt_eng1,n1_speed_eng1,n2_speed_eng1,oil_pressure_eng1,oil_temp_eng1,fuel_flow_eng1,vibration_n1_eng1,vibration_n2_eng1,epr_eng1,ff_ratio_eng1,bleed_pressure_eng1,bleed_temp_eng1,thrust_output_eng1,starter_valve_pos_eng1,reverser_pos_eng1,egt_eng2,n1_speed_eng2,n2_speed_eng2,oil_pressure_eng2,oil_temp_eng2,fuel_flow_eng2,vibration_n1_eng2,vibration_n2_eng2,epr_eng2,ff_ratio_eng2,bleed_pressure_eng2,bleed_temp_eng2,thrust_output_eng2,starter_valve_pos_eng2,reverser_pos_eng2,altitude,indicated_airspeed,ground_speed,mach_number,vertical_speed,heading,pitch,roll,yaw,angle_of_attack,latitude,longitude,weight,cg_position,flap_position,slat_position,spoiler_position,gear_position,autopilot_engaged,throttle_position,hyd_press_sys1,hyd_press_sys2,hyd_press_sys3,hyd_temp_sys1,hyd_temp_sys2,hyd_temp_sys3,hyd_fluid_level_1,hyd_fluid_level_2,hyd_fluid_level_3,hyd_pump_status,gen_voltage_1,gen_voltage_2,gen_freq_1,gen_freq_2,bus_voltage_ac,bus_voltage_dc,battery_voltage,battery_temp,battery_charge,apu_gen_voltage,oat,tat,wind_speed,wind_direction,baro_pressure,humidity,icing_indicator,turbulence_index,pack_flow_1,pack_flow_2,cabin_pressure,cabin_temp,cabin_altitude,diff_pressure,outflow_valve_pos,pressurization_rate,fuel_qty_total,fuel_qty_left,fuel_qty_right,fuel_qty_center,fuel_temp,fuel_pressure,fuel_imbalance,brake_temp_l1,brake_temp_l2,brake_temp_r1,brake_temp_r2,tire_press_nose,tire_press_left,tire_press_right,gear_actuator_press,aileron_pos_l,aileron_pos_r,elevator_pos,rudder_pos,trim_pos,stick_force_pitch,stick_force_roll,rudder_pedal_force
FLT_DEMO_0001,SkyWest Global,B737-800,N57382,JFK,LAX,2025-09-28T08:06:08.938Z,2025-09-28,TAXI_OUT,1,ENGINE_DEGRADATION,LOW,608.738803933438,54.7006524268266,74.62421240216285,45.25158263488528,88.9344853742723,2212.053034788505,0.33371643622578095,0.258145357472366,1.4418024727789684,0.9267731853795774,30.18893963877611,177.33621343568075,45.49283703092555,100,0.0,613.403044242793,55.1863357724265,75.03597024876144,46.24344214621053,90.93197373512879,2275.8716585869474,0.24310326398247123,0.19018547828988375,1.4757824123702097,0.9670290020244301,31.774670153034027,182.32993433782198,45.394484615168494,100,0.0,4803.342525865306,86.58285218103755,94.10816152776366,0.1322009395427041,472.88744773618674,257.9501652144077,3.0598710776504014,0.21065973811829636,0.5785092678764825,6.003416572521499,40.641179082360026,-73.78521743720262,109904.87361748583,23.83063623913202,8.777288637610125,7.5522171703558865,1.3761085156697914,63.36519892961192,0,44.903030303030306,2994.014422398836,2975.006874813311,2978.0010347747643,56.04192130226598,52.86306865338472,51.84261060157768,94.01569905259386,93.06525579422129,94.64868524271014,1,115.85292365143722,114.81646060913954,400.31784693270583,399.4840220022103,115.07132519330443,28.094209082733645,26.47081349455274,26.055995727340886,96.34570304930442,114.72811315503343,-6.65920042479986,-2.2575595385747786,27.265473470243332,229.60877382119727,842.8237420008261,65.5170142217506,1,0.11480283755063915,71.7585719062077,72.90629014864102,14.125833848860028,22.20858107028495,1459.697777777778,0.6103836914600551,56.757370478025635,56.88276834452477,24179.072195846882,8742.387138764529,8666.544842245226,6770.140214837128,-1.3755858713604834,28.20404996768402,75.84229651930218,92.50512297433033,86.55194503242691,82.7265863280444,77.72228920356432,188.3181745040561,194.6328176305038,195.77013835212594,2881.377985604843,0.44292046659453993,-0.2843713658164116,3.1700328592959535,0.8765630311334472,-2.088398450485047,36.77734611376716,12.212891663696553,17.668683276400575
FLT_DEMO_0001,SkyWest Global,B737-800,N57382,JFK,LAX,2025-09-28T08:06:20.938Z,2025-09-28,TAKEOFF,2,ENGINE_DEGRADATION,LOW,647.802720363227,61.164506807710744,78.1122207081851,48.14287601166871,94.14696113244443,2543.350290645004,0.25993750869832855,0.203850748603636

        flight_id      flight_phase        anomaly_type anomaly_severity
0   FLT_DEMO_0001          TAXI_OUT  ENGINE_DEGRADATION              LOW
1   FLT_DEMO_0001           TAKEOFF  ENGINE_DEGRADATION              LOW
2   FLT_DEMO_0001             CLIMB  ENGINE_DEGRADATION              LOW
6   FLT_DEMO_0001            CRUISE  ENGINE_DEGRADATION              LOW
20  FLT_DEMO_0001           DESCENT  ENGINE_DEGRADATION              LOW
24  FLT_DEMO_0001  APPROACH_LANDING  ENGINE_DEGRADATION              LOW


In [0]:
def build_flight_manifest(num_flights: int = FLIGHTS_HISTORICAL, anomaly_rate: float = ANOMALY_RATE) -> List[Dict[str, Any]]:
    manifest_rng = np.random.default_rng(RANDOM_SEED + 200)
    manifest_records: List[Dict[str, Any]] = []
    airline_cycle = cycle(AIRLINES)

    for flight_number in range(1, num_flights + 1):
        airline = next(airline_cycle)
        aircraft = AIRLINE_FLEET_MAP[airline][int(manifest_rng.integers(0, len(AIRLINE_FLEET_MAP[airline])))]
        route = AIRLINE_ROUTE_MAP[airline][int(manifest_rng.integers(0, len(AIRLINE_ROUTE_MAP[airline])))]
        departure_time = utc_now_naive() - timedelta(minutes=int(manifest_rng.integers(0, 365 * 24 * 60)))
        anomaly_config = sample_anomaly_config(manifest_rng) if float(manifest_rng.random()) < anomaly_rate else None
        manifest_records.append(
            {
                "flight_id": f"FLT_{flight_number:05d}",
                "airline": airline,
                "aircraft": aircraft,
                "route": route,
                "departure_time": departure_time,
                "arrival_time": departure_time + timedelta(hours=estimate_block_hours(route)),
                "anomaly_config": anomaly_config,
                "has_anomaly": anomaly_config is not None,
            }
        )
    return manifest_records


FLIGHT_MANIFEST_RECORDS = build_flight_manifest()
FLIGHTS_METADATA_PDF = pd.DataFrame(
    [
        {
            "flight_id": item["flight_id"],
            "airline": item["airline"],
            "aircraft_type": item["aircraft"]["aircraft_type"],
            "tail_number": item["aircraft"]["tail_number"],
            "origin": item["route"]["origin"],
            "destination": item["route"]["destination"],
            "departure_time": item["departure_time"],
            "arrival_time": item["arrival_time"],
            "has_anomaly": item["has_anomaly"],
            "anomaly_type": item["anomaly_config"]["anomaly_type"] if item["anomaly_config"] else None,
            "anomaly_severity": item["anomaly_config"]["anomaly_severity"] if item["anomaly_config"] else None,
        }
        for item in FLIGHT_MANIFEST_RECORDS
    ]
)

spark.createDataFrame(AIRCRAFT_REGISTRY_PDF).write.saveAsTable(AIRCRAFT_REGISTRY_TABLE_FQN, mode="overwrite")
spark.createDataFrame(FLIGHTS_METADATA_PDF).write.saveAsTable(FLIGHTS_METADATA_TABLE_FQN, mode="overwrite")


def generate_flight_from_manifest(record: Dict[str, Any]) -> pd.DataFrame:
    return generate_flight_data(
        flight_id=record["flight_id"],
        airline=record["airline"],
        aircraft=record["aircraft"],
        route=record["route"],
        anomaly_config=record["anomaly_config"],
        departure_time=record["departure_time"],
        n_records=RECORDS_PER_FLIGHT,
        random_state=(abs(hash(record["flight_id"])) + RANDOM_SEED) % (2**32),
    )


batch_size = 20
batch_summaries: List[Dict[str, Any]] = []
first_write = True

for batch_start in range(0, len(FLIGHT_MANIFEST_RECORDS), batch_size):
    batch_records = FLIGHT_MANIFEST_RECORDS[batch_start: batch_start + batch_size]
    with ThreadPoolExecutor(max_workers=min(8, len(batch_records))) as executor:
        batch_frames = list(executor.map(generate_flight_from_manifest, batch_records))

    batch_pdf = pd.concat(batch_frames, ignore_index=True)
    batch_sdf = spark.createDataFrame(batch_pdf)
    batch_sdf.write.saveAsTable(
        RAW_TABLE_FQN,
        mode="overwrite" if first_write else "append",
        partitionBy=["airline", "event_date"],
    )
    first_write = False

    batch_summaries.append(
        {
            "batch_number": len(batch_summaries) + 1,
            "flights_written": len(batch_records),
            "records_written": len(batch_pdf),
            "anomalous_flights": int(sum(1 for record in batch_records if record["has_anomaly"])),
            "min_timestamp": batch_pdf["timestamp"].min(),
            "max_timestamp": batch_pdf["timestamp"].max(),
        }
    )
    print(
        f"Wrote batch {len(batch_summaries):02d} with {len(batch_records):,} flights and {len(batch_pdf):,} telemetry rows"
    )

historical_summary_pdf = pd.DataFrame(batch_summaries)
raw_record_count = spark.table(RAW_TABLE_FQN).count()
metadata_record_count = spark.table(FLIGHTS_METADATA_TABLE_FQN).count()
aircraft_record_count = spark.table(AIRCRAFT_REGISTRY_TABLE_FQN).count()

summary_pdf = pd.DataFrame(
    [
        {
            "raw_record_count": raw_record_count,
            "metadata_record_count": metadata_record_count,
            "aircraft_record_count": aircraft_record_count,
            "anomalous_flights": int(FLIGHTS_METADATA_PDF["has_anomaly"].sum()),
            "distinct_airlines": FLIGHTS_METADATA_PDF["airline"].nunique(),
            "distinct_tail_numbers": FLIGHTS_METADATA_PDF["tail_number"].nunique(),
        }
    ]
)

display(historical_summary_pdf.head(10))
display(summary_pdf)
display(spark.table(RAW_TABLE_FQN).select("flight_id", "airline", "origin", "destination", "timestamp", "flight_phase", "anomaly_type", "anomaly_severity").limit(5))


Wrote batch 01 with 20 flights and 20,000 telemetry rows
Wrote batch 02 with 20 flights and 20,000 telemetry rows
Wrote batch 03 with 20 flights and 20,000 telemetry rows
Wrote batch 04 with 20 flights and 20,000 telemetry rows
Wrote batch 05 with 20 flights and 20,000 telemetry rows
Wrote batch 06 with 20 flights and 20,000 telemetry rows
Wrote batch 07 with 20 flights and 20,000 telemetry rows
Wrote batch 08 with 20 flights and 20,000 telemetry rows
Wrote batch 09 with 20 flights and 20,000 telemetry rows
Wrote batch 10 with 20 flights and 20,000 telemetry rows
Wrote batch 11 with 20 flights and 20,000 telemetry rows
Wrote batch 12 with 20 flights and 20,000 telemetry rows
Wrote batch 13 with 20 flights and 20,000 telemetry rows
Wrote batch 14 with 20 flights and 20,000 telemetry rows
Wrote batch 15 with 20 flights and 20,000 telemetry rows
Wrote batch 16 with 20 flights and 20,000 telemetry rows
Wrote batch 17 with 20 flights and 20,000 telemetry rows
Wrote batch 18 with 20 flights 

batch_number,flights_written,records_written,anomalous_flights,min_timestamp,max_timestamp
1,20,20000,1,2025-10-20T22:38:42.358Z,2026-08-11T10:38:30.358Z
2,20,20000,3,2025-09-11T19:12:42.358Z,2026-08-04T13:42:09.688Z
3,20,20000,4,2025-10-12T18:46:42.359Z,2026-08-19T08:27:46.431Z
4,20,20000,1,2025-10-02T15:23:42.359Z,2026-07-13T06:59:30.359Z
5,20,20000,2,2025-08-31T22:53:42.360Z,2026-08-21T06:32:35.217Z
6,20,20000,2,2025-09-02T03:13:42.360Z,2026-08-23T04:37:30.360Z
7,20,20000,4,2025-09-10T03:39:42.361Z,2026-08-26T20:04:30.361Z
8,20,20000,3,2025-10-06T09:52:42.362Z,2026-08-31T08:25:30.362Z
9,20,20000,2,2025-09-21T08:39:42.362Z,2026-08-25T23:13:27.161Z
10,20,20000,0,2025-09-05T13:44:42.362Z,2026-08-25T21:29:22.474Z


raw_record_count,metadata_record_count,aircraft_record_count,anomalous_flights,distinct_airlines,distinct_tail_numbers
1000000,1000,328,103,10,309


flight_id,airline,origin,destination,timestamp,flight_phase,anomaly_type,anomaly_severity
FLT_00012,Atlas Pacific,ORD,DFW,2026-01-14T23:34:42.358Z,TAXI_OUT,null,null
FLT_00012,Atlas Pacific,ORD,DFW,2026-01-14T23:34:49.809Z,TAXI_OUT,null,null
FLT_00012,Atlas Pacific,ORD,DFW,2026-01-14T23:34:57.261Z,TAXI_OUT,null,null
FLT_00012,Atlas Pacific,ORD,DFW,2026-01-14T23:35:04.712Z,TAXI_OUT,null,null
FLT_00012,Atlas Pacific,ORD,DFW,2026-01-14T23:35:12.164Z,TAXI_OUT,null,null


In [0]:
API_HOST = "127.0.0.1"
API_PORT = 8008
API_STATE: Dict[str, Any] = {"framework": None, "server": None, "thread": None, "started_at": None}


def find_available_port(preferred_port: int = API_PORT) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        if sock.connect_ex((API_HOST, preferred_port)) != 0:
            return preferred_port
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((API_HOST, 0))
        return int(sock.getsockname()[1])


def build_live_batch(batch_flights: int = 1, records_per_flight: int = 50) -> pd.DataFrame:
    rng = np.random.default_rng(int(time.time()))
    frames: List[pd.DataFrame] = []
    for batch_index in range(batch_flights):
        airline = AIRLINES[int(rng.integers(0, len(AIRLINES)))]
        aircraft = AIRLINE_FLEET_MAP[airline][int(rng.integers(0, len(AIRLINE_FLEET_MAP[airline])))]
        route = AIRLINE_ROUTE_MAP[airline][int(rng.integers(0, len(AIRLINE_ROUTE_MAP[airline])))]
        anomaly_config = sample_anomaly_config(rng) if float(rng.random()) < ANOMALY_RATE else None
        frames.append(
            generate_flight_data(
                flight_id=f"LIVE_{int(time.time())}_{batch_index:02d}",
                airline=airline,
                aircraft=aircraft,
                route=route,
                anomaly_config=anomaly_config,
                departure_time=utc_now_naive(),
                n_records=records_per_flight,
                random_state=int(rng.integers(0, 2**32 - 1)),
            )
        )
    return pd.concat(frames, ignore_index=True)


def normalize_ingest_payload(payload: Any) -> pd.DataFrame:
    if isinstance(payload, dict) and "records" in payload:
        records = payload["records"]
    elif isinstance(payload, list):
        records = payload
    else:
        records = [payload]

    payload_pdf = pd.json_normalize(records)
    if payload_pdf.empty:
        raise ValueError("No telemetry records were provided to the API")

    for column_name in METADATA_COLUMNS + PARAMETER_NAMES:
        if column_name not in payload_pdf.columns:
            payload_pdf[column_name] = np.nan if column_name in PARAMETER_NAMES else None

    payload_pdf["timestamp"] = pd.to_datetime(payload_pdf["timestamp"], errors="coerce")
    payload_pdf["timestamp"] = payload_pdf["timestamp"].fillna(pd.Timestamp.utcnow())
    payload_pdf["event_date"] = pd.to_datetime(payload_pdf["timestamp"]).dt.date
    payload_pdf["record_sequence"] = payload_pdf["record_sequence"].fillna(0).astype(int)
    payload_pdf["autopilot_engaged"] = payload_pdf["autopilot_engaged"].fillna(0).astype(int)
    payload_pdf["hyd_pump_status"] = payload_pdf["hyd_pump_status"].fillna(1).astype(int)
    payload_pdf["icing_indicator"] = payload_pdf["icing_indicator"].fillna(0).astype(int)
    payload_pdf["airline"] = payload_pdf["airline"].fillna("API_INGEST")
    payload_pdf["flight_phase"] = payload_pdf["flight_phase"].fillna("EXTERNAL_INGEST")

    return payload_pdf[METADATA_COLUMNS + PARAMETER_NAMES]


def ingest_records_to_staging(payload: Any, source: str) -> Dict[str, Any]:
    payload_pdf = normalize_ingest_payload(payload)
    payload_pdf["flight_phase"] = payload_pdf["flight_phase"].fillna(source)
    spark.createDataFrame(payload_pdf).write.saveAsTable(API_STAGING_TABLE_FQN, mode="append", partitionBy=["airline", "event_date"])
    return {
        "records_written": int(len(payload_pdf)),
        "flights": int(payload_pdf["flight_id"].nunique()),
        "min_timestamp": str(payload_pdf["timestamp"].min()),
        "max_timestamp": str(payload_pdf["timestamp"].max()),
        "target_table": API_STAGING_TABLE_FQN,
    }


def start_ingest_api(port: Optional[int] = None) -> str:
    desired_port = port or find_available_port(API_PORT)
    if API_STATE.get("thread") and API_STATE["thread"].is_alive():
        return f"http://{API_HOST}:{API_STATE['port']}"

    try:
        from flask import Flask, jsonify, request
        from werkzeug.serving import make_server

        app = Flask("aircraft_data_simulator_api")

        @app.get("/api/v1/health")
        def health() -> Any:
            return jsonify(
                {
                    "status": "ok",
                    "framework": "flask",
                    "raw_table": RAW_TABLE_FQN,
                    "staging_table": API_STAGING_TABLE_FQN,
                }
            )

        @app.post("/api/v1/flight-data")
        def post_flight_data() -> Any:
            result = ingest_records_to_staging(request.get_json(force=True), source="api_post")
            return jsonify(result)

        @app.get("/api/v1/simulate")
        def simulate() -> Any:
            batch_flights = int(request.args.get("batch_flights", 1))
            records_per_flight = int(request.args.get("records_per_flight", 25))
            batch_pdf = build_live_batch(batch_flights=batch_flights, records_per_flight=records_per_flight)
            payload = json.loads(batch_pdf.to_json(orient="records", date_format="iso"))
            result = ingest_records_to_staging(payload, source="api_simulate")
            return jsonify({"records": payload, "ingest_result": result})

        server = make_server(API_HOST, desired_port, app)
        thread = threading.Thread(target=server.serve_forever, daemon=True)
        thread.start()
        API_STATE.update({"framework": "flask", "server": server, "thread": thread, "port": desired_port, "started_at": utc_now_naive()})
    except ImportError:
        from fastapi import FastAPI, Request
        import uvicorn

        app = FastAPI(title="aircraft_data_simulator_api")

        @app.get("/api/v1/health")
        async def health() -> Dict[str, Any]:
            return {
                "status": "ok",
                "framework": "fastapi",
                "raw_table": RAW_TABLE_FQN,
                "staging_table": API_STAGING_TABLE_FQN,
            }

        @app.post("/api/v1/flight-data")
        async def post_flight_data(request: Request) -> Dict[str, Any]:
            payload = await request.json()
            return ingest_records_to_staging(payload, source="api_post")

        @app.get("/api/v1/simulate")
        async def simulate(batch_flights: int = 1, records_per_flight: int = 25) -> Dict[str, Any]:
            batch_pdf = build_live_batch(batch_flights=batch_flights, records_per_flight=records_per_flight)
            payload = json.loads(batch_pdf.to_json(orient="records", date_format="iso"))
            result = ingest_records_to_staging(payload, source="api_simulate")
            return {"records": payload, "ingest_result": result}

        config = uvicorn.Config(app, host=API_HOST, port=desired_port, log_level="warning")
        server = uvicorn.Server(config=config)
        thread = threading.Thread(target=server.run, daemon=True)
        thread.start()
        API_STATE.update({"framework": "fastapi", "server": server, "thread": thread, "port": desired_port, "started_at": utc_now_naive()})

    time.sleep(2)
    return f"http://{API_HOST}:{desired_port}"


def stop_ingest_api() -> None:
    if not API_STATE.get("server"):
        print("API server is not running")
        return
    if API_STATE["framework"] == "flask":
        API_STATE["server"].shutdown()
    else:
        API_STATE["server"].should_exit = True
    API_STATE["server"] = None
    API_STATE["thread"] = None
    print("API server stopped")


def call_json_endpoint(url: str, method: str = "GET", payload: Optional[Any] = None) -> Any:
    from urllib import request as urllib_request

    headers = {"Content-Type": "application/json"}
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request_obj = urllib_request.Request(url, data=data, headers=headers, method=method)
    with urllib_request.urlopen(request_obj, timeout=60) as response:
        return json.loads(response.read().decode("utf-8"))


api_base_url = start_ingest_api()
health_response = call_json_endpoint(f"{api_base_url}/api/v1/health")
demo_batch_pdf = build_live_batch(batch_flights=1, records_per_flight=5)
demo_payload = json.loads(demo_batch_pdf.to_json(orient="records", date_format="iso"))
post_response = call_json_endpoint(f"{api_base_url}/api/v1/flight-data", method="POST", payload=demo_payload)
simulate_response = call_json_endpoint(f"{api_base_url}/api/v1/simulate?batch_flights=1&records_per_flight=3")

api_demo_pdf = pd.DataFrame(
    [
        {"endpoint": "/api/v1/health", "response": json.dumps(health_response)[:300]},
        {"endpoint": "/api/v1/flight-data", "response": json.dumps(post_response)[:300]},
        {"endpoint": "/api/v1/simulate", "response": json.dumps(simulate_response["ingest_result"])[:300]},
    ]
)
display(api_demo_pdf)
display(spark.table(API_STAGING_TABLE_FQN).select("flight_id", "airline", "timestamp", "flight_phase", "anomaly_type", "anomaly_severity").limit(5))
print(f"API available at {api_base_url}")


endpoint,response
/api/v1/health,"{""status"": ""ok"", ""framework"": ""fastapi"", ""raw_table"": ""genie_zeroops_mfg_catalog.aircraft_maintenance.raw_flight_telemetry"", ""staging_table"": ""genie_zeroops_mfg_catalog.aircraft_maintenance.staging_flight_telemetry_api""}"
/api/v1/flight-data,"{""records_written"": 5, ""flights"": 1, ""min_timestamp"": ""2026-08-31 14:29:37.377000"", ""max_timestamp"": ""2026-08-31 14:30:25.377000"", ""target_table"": ""genie_zeroops_mfg_catalog.aircraft_maintenance.staging_flight_telemetry_api""}"
/api/v1/simulate,"{""records_written"": 3, ""flights"": 1, ""min_timestamp"": ""2026-08-31 14:29:38.752000"", ""max_timestamp"": ""2026-08-31 14:30:02.752000"", ""target_table"": ""genie_zeroops_mfg_catalog.aircraft_maintenance.staging_flight_telemetry_api""}"


flight_id,airline,timestamp,flight_phase,anomaly_type,anomaly_severity
LIVE_1788186577_00,Aurora Airlines,2026-08-31T14:29:37.377Z,CLIMB,null,null
LIVE_1788186577_00,Aurora Airlines,2026-08-31T14:30:13.377Z,CRUISE,null,null
LIVE_1788186577_00,Aurora Airlines,2026-08-31T14:29:49.377Z,CRUISE,null,null
LIVE_1788186577_00,Aurora Airlines,2026-08-31T14:30:01.377Z,CRUISE,null,null
LIVE_1788186577_00,Aurora Airlines,2026-08-31T14:30:25.377Z,DESCENT,null,null


API available at http://127.0.0.1:54565


In [0]:
def append_live_batch_to_raw(batch_flights: int = 2, records_per_flight: int = 120) -> Dict[str, Any]:
    live_pdf = build_live_batch(batch_flights=batch_flights, records_per_flight=records_per_flight)
    spark.createDataFrame(live_pdf).write.saveAsTable(RAW_TABLE_FQN, mode="append", partitionBy=["airline", "event_date"])
    return {
        "records_written": int(len(live_pdf)),
        "flights_simulated": int(live_pdf["flight_id"].nunique()),
        "min_timestamp": str(live_pdf["timestamp"].min()),
        "max_timestamp": str(live_pdf["timestamp"].max()),
    }


def run_streaming_simulation(
    iterations: int = 3,
    interval_seconds: int = 10,
    batch_flights: int = 2,
    records_per_flight: int = 120,
) -> pd.DataFrame:
    results: List[Dict[str, Any]] = []
    for iteration in range(1, iterations + 1):
        write_result = append_live_batch_to_raw(
            batch_flights=batch_flights,
            records_per_flight=records_per_flight,
        )
        write_result["iteration"] = iteration
        results.append(write_result)
        print(
            f"Iteration {iteration}: wrote {write_result['records_written']:,} rows across {write_result['flights_simulated']} flights"
        )
        if iteration < iterations:
            time.sleep(interval_seconds)

    result_pdf = pd.DataFrame(results)
    display(result_pdf)
    display(
        spark.table(RAW_TABLE_FQN)
        .select("flight_id", "airline", "timestamp", "flight_phase", "anomaly_type", "anomaly_severity")
        .orderBy("timestamp", ascending=False)
        .limit(5)
    )
    return result_pdf


print("Call run_streaming_simulation(iterations=3, interval_seconds=10, batch_flights=2, records_per_flight=120) to simulate live append-only telemetry.")
streaming_validation_pdf = run_streaming_simulation(iterations=1, interval_seconds=1, batch_flights=1, records_per_flight=10)


Call run_streaming_simulation(iterations=3, interval_seconds=10, batch_flights=2, records_per_flight=120) to simulate live append-only telemetry.
Iteration 1: wrote 10 rows across 1 flights


records_written,flights_simulated,min_timestamp,max_timestamp,iteration
10,1,2026-08-31 14:29:47.813424,2026-08-31 14:31:35.813424,1


flight_id,airline,timestamp,flight_phase,anomaly_type,anomaly_severity
LIVE_1788186587_00,Atlas Pacific,2026-08-31T14:31:35.813Z,DESCENT,null,null
LIVE_1788186587_00,Atlas Pacific,2026-08-31T14:31:23.813Z,DESCENT,null,null
LIVE_1788186587_00,Atlas Pacific,2026-08-31T14:31:11.813Z,CRUISE,null,null
LIVE_1788186587_00,Atlas Pacific,2026-08-31T14:30:59.813Z,CRUISE,null,null
LIVE_1788186587_00,Atlas Pacific,2026-08-31T14:30:47.813Z,CRUISE,null,null
